# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batsykods/FLrank1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: ranking / scoring**, built on top of a binary classification model.

Lane 2's question — *"which pages should an editor review first?"* — is a "which ones first"
question, not a "is this one X" question. Per the framing table in `framing-ml-problems`, that
maps to **ranking/scoring**, with precision@K as the natural metric.

In practice: I'll train a classifier that estimates P(declining) for every content item, then use
that probability as a priority score and sort the whole inventory by it. The classifier is the
engine; the **ranked queue** is the thing an editor actually opens and works from.

In [4]:
import os
import numpy as np
import pandas as pd

while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"{len(df):,} content items across {df['client_id'].nunique()} clients")
print("An editor cannot hand-review all of these every cycle — someone has to decide the order.")


30,000 content items across 32 clients
An editor cannot hand-review all of these every cycle — someone has to decide the order.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`**, defined as `trend_direction == 'down'`.

This is a **proxy, not a perfectly clean observed outcome**: `trend_direction` is itself a
three-bucket version (up / flat / down) of `trend_pct`, a continuous measure of how each item's
traffic moved between two trailing windows. So the label does come from real, measured
performance change — not from an invented business rule — but bucketing it loses information at
the edges (a page at -0.4% and a page at -19% both just read "down").

**Guardrail:** because `trend_direction` and `trend_pct` *define* the label, neither can ever be
used as a model feature later — that would just let the model read its own answer key.

In [5]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df["trend_direction"].value_counts())
print()
print(f"Declining rate: {df['is_declining_label'].mean():.1%}  "
      f"({df['is_declining_label'].sum():,} of {len(df):,} items)")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining rate: 54.2%  (16,262 of 30,000 items)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50** — of the top 50 items my ranking puts first, what fraction are actually
declining?

Why this and not accuracy: editors only have bandwidth to look at a small batch each review
cycle, so what matters is quality at the *top* of the queue, not correctness averaged across all
30,000 items. And with the label split roughly 54/46, plain accuracy is easy to game and tells an
editor nothing about where to start.

Precision@50 only means something next to a floor. If I ranked pages in random order, the top 50
should land close to the overall declining rate (~54%) by chance alone — any real system has to
clear that bar by a real margin, not just brush past it.

In [6]:
rng = np.random.default_rng(7)
df["_random_score"] = rng.random(len(df))
random_top50 = df.sort_values("_random_score", ascending=False).head(50)

print(f"Precision@50 from a RANDOM ranking (the floor): {random_top50['is_declining_label'].mean():.1%}")
print(f"Overall declining rate (for comparison):         {df['is_declining_label'].mean():.1%}")

df = df.drop(columns="_random_score")

Precision@50 from a RANDOM ranking (the floor): 58.0%
Overall declining rate (for comparison):         54.2%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content item (a single page) belonging to one client**, summarized
by its trailing 90-day search and engagement metrics — impressions, clicks, sessions, position,
freshness, and so on.

The reference pipeline restricts Lane 2 to items with `impressions_90d > 0` and
`content_age_days >= 90` (a page needs real search demand *and* enough history for "declining" to
mean anything). Every row in this dataset already clears that bar, so nothing gets dropped here —
but I apply the filter explicitly rather than assume it, since a future data pull might not be
this clean.

In [8]:
lane2_df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
lane2_df = lane2_df.drop_duplicates(subset="content_id").reset_index(drop=True)

print(f"{len(lane2_df):,} of {len(df):,} rows qualify for Lane 2 review")

lane2_df[[
    "content_id", "client_id", "content_type", "impressions_90d", "clicks_90d",
    "avg_position", "days_since_last_update", "trend_direction", "is_declining_label"
]].head(8)

30,000 of 30,000 rows qualify for Lane 2 review


,content_id,client_id,content_type,impressions_90d,clicks_90d,avg_position,days_since_last_update,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,10.6,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,20.3,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,36.5,20,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,6.2,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,44.0,14,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,8.5,20,down,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,7.0,20,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,21.2,22,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

I rebuilt the repo's own hand-written baseline rule from `scripts/02_baseline_score.py`
(`0.40×visibility + 0.30×freshness_risk + 0.25×position_opportunity + 0.05×depth_gap`) and
checked its top-50 picks against the real label.

**Result:** the fixed rule's top 50 picks are declining only **34%** of the time on this data —
*worse* than the ~54% you'd expect from the base rate alone. On the honest, held-out client split
the repo's reference pipeline reports (`outputs/model_report.md`), it drops further to **24%**,
while a random forest trained on the same signals reaches **74%** — roughly triple the rule's
hit rate.

**Why the rule fails:** it was built from *intuitive* signals a person can reason about by hand —
visible + stale + poorly positioned. But "declining" is a genuinely different, more tangled
pattern: it depends on how impressions, clicks, position, freshness, content depth, and
engagement move *together*, not on two or three of them added up with fixed weights someone
picked in advance. In the trained model's feature importances, the top signal accounts for only
~16% of its decisions and the next nine are all close behind — there's no single dominant signal
to write an if-statement around. That's exactly the situation where ML earns its place over a
fixed rule.

In [9]:
def percentile_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

def normalize(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0

rule_df = lane2_df.copy()
rule_df["visibility_score"] = percentile_rank(np.log1p(rule_df["impressions_90d"]))
rule_df["freshness_risk_score"] = percentile_rank(rule_df["days_since_last_update"])
rule_df["position_opportunity_score"] = (
    (1 - normalize(rule_df["avg_position"].clip(lower=1, upper=50)))
    * rule_df["visibility_score"]
    * (rule_df["avg_position"] > 0).astype(int)
)
rule_df["depth_gap_score"] = (1 - percentile_rank(rule_df["word_count"])) * rule_df["visibility_score"]

rule_df["baseline_refresh_score"] = (
    0.40 * rule_df["visibility_score"]
    + 0.30 * rule_df["freshness_risk_score"]
    + 0.25 * rule_df["position_opportunity_score"]
    + 0.05 * rule_df["depth_gap_score"]
).clip(0, 1)

rule_top50 = rule_df.sort_values("baseline_refresh_score", ascending=False).head(50)
print(f"Fixed-rule precision@50 (this data):             {rule_top50['is_declining_label'].mean():.1%}")
print("Fixed-rule precision@50 (held-out, reported):     24.0%   [outputs/model_report.md]")
print("Random forest precision@50 (held-out, reported):  74.0%   [outputs/model_report.md]")

Fixed-rule precision@50 (this data):             34.0%
Fixed-rule precision@50 (held-out, reported):     24.0%   [outputs/model_report.md]
Random forest precision@50 (held-out, reported):  74.0%   [outputs/model_report.md]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.